# 03 · 表格数据 Transformer：特征分词与 FT-Transformer  Tabular Transformer: FeatureTokenizer & FT-Transformer

> **能力标签**：GA1（Transformer 架构 / Transformer Architecture）

## 目标 Objectives

完成本课后，你应该能够：

1. 解释为什么将 Transformer 直接应用于**表格数据**（tabular data）需要特殊的特征处理方式。
2. 实现 **FeatureTokenizer**：将每个数值特征映射到 $d_\text{token}$ 维 token，并前置一个可学习的 **[CLS] token**。
3. 描述 **FT-Transformer**（Feature Tokenizer + Transformer）的整体架构。
4. 解释 [CLS] token 作为"全局表征"如何用于分类预测头（classification head）。

> 对应能力：**GA1**

## 概念讲解 Concepts

### 1. 表格数据的挑战 Tabular Data Challenges

传统深度学习在图像（CNN）和文本（Transformer）上表现出色，但**表格数据**（如传感器读数、销售记录、问卷数据）有以下特点：

| 特点 | 说明 |
|------|------|
| **混合特征类型** | 数值型（连续）、类别型（离散）混合 |
| **特征间无空间结构** | 列顺序不像像素或 token 那样有天然先验 |
| **高维稀疏** | 医疗数据中常见大量缺失值 |

为此，**FT-Transformer**（Gorishniy et al., 2021）提出将每个特征嵌入为独立的 token，再经过 Transformer 建模特征间交互。

---

### 2. FeatureTokenizer 设计 Feature Tokenizer

对于 $n$ 个数值特征，每个特征 $j$ 学习独立的权重向量 $\mathbf{w}_j \in \mathbb{R}^{d}$ 和偏置向量 $\mathbf{b}_j \in \mathbb{R}^{d}$：

$$\text{token}_j = x_j \cdot \mathbf{w}_j + \mathbf{b}_j \quad \in \mathbb{R}^d$$

对 batch 中所有样本：输入 $X \in \mathbb{R}^{B \times n}$ → 输出 $T \in \mathbb{R}^{B \times n \times d}$

**[CLS] token**（可学习参数 $\mathbf{c} \in \mathbb{R}^{d}$）前置到序列头部：

$$\text{输出} = [\mathbf{c}; \text{token}_1; \ldots; \text{token}_n] \in \mathbb{R}^{B \times (n+1) \times d}$$

---

### 3. FT-Transformer 整体架构 Full Architecture

```
X (B, n) ──→ FeatureTokenizer ──→ (B, n+1, d) ──→ Transformer Block(s) ──→ CLS token ──→ 分类头
                                    ↑[CLS]                                     [0, :, :]
```

与 BERT/ViT 类似：[CLS] token 经过 Transformer 块后的输出被提取出来，送入一个线性分类头（如 MLP）做最终预测。


## 示例 Worked Example

In [ ]:
# ── FeatureTokenizer 完整实现（镜像 w7-ft-transformer）──────────────────────
import torch
import torch.nn as nn

torch.manual_seed(42)


class FeatureTokenizer(nn.Module):
    """FT-Transformer 特征分词器：每个数值特征映射到一个 d_token 维 token，
    并前置一个可学习的 [CLS] token。
    FT-Transformer feature tokenizer: each numeric feature → d_token-dim token,
    with a prepended learnable [CLS] token.
    """
    def __init__(self, n_features, d_token):
        super().__init__()
        # 每个特征独立的权重和偏置 (n_features, d_token)
        self.weight = nn.Parameter(torch.randn(n_features, d_token))
        self.bias   = nn.Parameter(torch.zeros(n_features, d_token))
        # 可学习的 [CLS] token (1, d_token)
        self.cls    = nn.Parameter(torch.randn(1, d_token))

    def forward(self, x):          # x: (B, n_features)
        # 特征嵌入：x_j * w_j + b_j → (B, n_features, d_token)
        tokens = x.unsqueeze(-1) * self.weight + self.bias
        # 将 [CLS] 广播到 batch，前置拼接
        cls = self.cls.expand(x.shape[0], 1, -1)        # (B, 1, d_token)
        return torch.cat([cls, tokens], dim=1)           # (B, n_features+1, d_token)


# ── 形状验证 Shape verification ───────────────────────────────────────────────
B, n_features, d_token = 4, 10, 32
tokenizer = FeatureTokenizer(n_features=n_features, d_token=d_token)

x = torch.randn(B, n_features)
out = tokenizer(x)

print("FeatureTokenizer 前向传播:")
print(f"  输入 x.shape         = {x.shape}      (B={B}, n_features={n_features})")
print(f"  输出 out.shape       = {out.shape}  (B, n_features+1={n_features+1}, d_token={d_token})")
print(f"  [CLS] token 位置     = out[:, 0, :]  形状 {out[:, 0, :].shape}")
print(f"  特征 token 位置      = out[:, 1:, :] 形状 {out[:, 1:, :].shape}")

assert out.shape == (B, n_features + 1, d_token), f"期望 ({B}, {n_features+1}, {d_token})，得到 {out.shape}"
print("✓ 输出形状正确 Output shape OK")


In [ ]:
# ── 参数结构解析 Parameter Analysis ─────────────────────────────────────────
print("FeatureTokenizer 参数 Parameters:")
for name, param in tokenizer.named_parameters():
    print(f"  {name:10s}: shape={param.shape}, numel={param.numel()}")

total_params = sum(p.numel() for p in tokenizer.parameters())
print(f"\n  总参数量 Total: {total_params}")
expected = n_features * d_token + n_features * d_token + d_token  # weight + bias + cls
print(f"  期望参数量 Expected: {n_features}×{d_token} + {n_features}×{d_token} + {d_token} = {expected}")
assert total_params == expected
print("✓ 参数量正确 Parameter count OK")


In [ ]:
# ── CLS token 作为全局表征：接入分类头 CLS Token → Classification Head ───────
# 在完整 FT-Transformer 中，FeatureTokenizer → Transformer Block(s) → CLS → 分类头

# 简化演示：直接用 FeatureTokenizer 的 CLS 输出接入 MLP（省略 Transformer 块）
class SimpleFTClassifier(nn.Module):
    """简化版 FT-Transformer 分类器（展示整体数据流）。
    Simplified FT-Transformer: FeatureTokenizer → [CLS] → Linear head.
    （生产版在 FeatureTokenizer 与分类头之间插入若干 TransformerBlock）
    """
    def __init__(self, n_features, d_token, n_classes):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_token)
        # 简化：跳过 Transformer 块，直接从 CLS token 做分类
        # 完整版：此处应插入 N 层 TransformerBlock
        self.head = nn.Linear(d_token, n_classes)

    def forward(self, x):              # x: (B, n_features)
        tokens = self.tokenizer(x)     # (B, n_features+1, d_token)
        cls_repr = tokens[:, 0, :]     # 取 [CLS] token  (B, d_token)
        return self.head(cls_repr)     # (B, n_classes)


torch.manual_seed(7)
model = SimpleFTClassifier(n_features=10, d_token=32, n_classes=2)
x_batch = torch.randn(8, 10)   # 8 个样本，10 个特征
logits = model(x_batch)

print(f"SimpleFTClassifier 前向传播:")
print(f"  输入  x_batch.shape = {x_batch.shape}")
print(f"  输出 logits.shape   = {logits.shape}  (应为 (8, 2))")
assert logits.shape == (8, 2)
print("✓ 分类头输出形状正确 Classifier head shape OK")


In [ ]:
# ── 验证 FeatureTokenizer 的线性结构 Verify Linear Tokenization ─────────────
# 每个特征 j 的 token = x_j * weight[j] + bias[j]
# 手工计算第 0 个样本、第 0 个特征的 token，与模型输出对比

sample_x = torch.randn(1, n_features)
with torch.no_grad():
    toks = tokenizer(sample_x)  # (1, n_features+1, d_token)

# 手工计算特征 0 的 token（index 1，因为 index 0 是 CLS）
feat_idx = 0
manual_token = sample_x[0, feat_idx] * tokenizer.weight[feat_idx] + tokenizer.bias[feat_idx]

err = (toks[0, feat_idx + 1] - manual_token).abs().max().item()
print(f"特征 {feat_idx} 手工 token vs. FeatureTokenizer 输出误差: {err:.2e}")
assert err < 1e-5, f"线性化验证失败 Linearity check failed: {err}"
print("✓ 每个特征 token = x_j * weight[j] + bias[j] 验证通过 Linearity OK")


## 动手 Exercise

In [ ]:
# ── 动手练习：支持缺失值的 FeatureTokenizer（零掩码策略）─────────────────────
# 表格数据中常有缺失值。零掩码策略：缺失特征的输入值置 0，同时对应 weight 置零，
# 使该特征不贡献到 token（只保留 bias，或完全归零）。
# 任务：实现 MaskedFeatureTokenizer，接受额外的 mask 张量
#       mask: (B, n_features)，1 表示有效，0 表示缺失
# 验证：缺失特征对应的 token 应等于 bias[j]（因为 x_j 被置 0）

class MaskedFeatureTokenizer(nn.Module):
    """带缺失值掩码的特征分词器。
    mask=1: valid feature (normal tokenization x*w + b)
    mask=0: missing feature (token = bias only, i.e., 0*w + b = b)
    """
    def __init__(self, n_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_features, d_token))
        self.bias   = nn.Parameter(torch.zeros(n_features, d_token))
        self.cls    = nn.Parameter(torch.randn(1, d_token))

    def forward(self, x, mask=None):   # x: (B, n_features), mask: (B, n_features)
        if mask is not None:
            # 缺失位置的 x 置 0（等效于令 token = bias[j]）
            x = x * mask   # (B, n_features)
        tokens = x.unsqueeze(-1) * self.weight + self.bias   # (B, n_features, d_token)
        cls = self.cls.expand(x.shape[0], 1, -1)
        return torch.cat([cls, tokens], dim=1)


torch.manual_seed(123)
n_f, d_t = 4, 8
mft = MaskedFeatureTokenizer(n_features=n_f, d_token=d_t)

x_test = torch.randn(2, n_f)
# 第 0 个样本：特征 1 缺失；第 1 个样本：特征 3 缺失
mask_test = torch.tensor([[1, 0, 1, 1],
                           [1, 1, 1, 0]], dtype=torch.float)

with torch.no_grad():
    out_masked = mft(x_test, mask=mask_test)

# 验证：缺失特征（mask=0）的 token 应等于对应的 bias
missing_token_s0_f1 = out_masked[0, 2, :]    # 样本 0，特征 1（index 2，因为 CLS 在 0）
expected_s0_f1 = mft.bias[1]
err_s0 = (missing_token_s0_f1 - expected_s0_f1).abs().max().item()

missing_token_s1_f3 = out_masked[1, 4, :]    # 样本 1，特征 3（index 4）
expected_s1_f3 = mft.bias[3]
err_s1 = (missing_token_s1_f3 - expected_s1_f3).abs().max().item()

print("MaskedFeatureTokenizer 缺失值验证:")
print(f"  样本 0 特征 1（缺失）token 误差 vs bias: {err_s0:.2e}")
print(f"  样本 1 特征 3（缺失）token 误差 vs bias: {err_s1:.2e}")
assert err_s0 < 1e-5 and err_s1 < 1e-5
print("✓ 缺失值零掩码策略正确 Missing-value zero-masking OK")


## 延伸阅读 Further Reading

1. **Gorishniy et al. "Revisiting Deep Learning Models for Tabular Data" (NeurIPS 2021)**：<https://arxiv.org/abs/2106.11959> — FT-Transformer 原始论文；提出 FeatureTokenizer + Transformer 的表格数据建模方案。
2. **Karpathy "Let's build GPT" (YouTube)**：<https://www.youtube.com/watch?v=kCc8FmEb1nY> — 从零构建 Transformer，[CLS] 类 token 的直觉来自于 BERT。
3. **The Annotated Transformer (Harvard NLP)**：<https://nlp.seas.harvard.edu/annotated-transformer/> — 含完整 Transformer 实现；理解 FeatureTokenizer 后阅读本文可快速上手完整架构。
4. **Devlin et al. "BERT" (2019)**：<https://arxiv.org/abs/1810.04805> — [CLS] token 作为句级表征的来源；FT-Transformer 的 [CLS] 设计直接借鉴 BERT。
5. **Huang et al. "TabTransformer" (2020)**：<https://arxiv.org/abs/2012.06678> — 另一篇表格 Transformer，专注类别型特征嵌入；与 FT-Transformer 互为对照。

## 关联练习 Related Assignments

```bash
python check.py 03-ft-transformer
```

> 实现 `FeatureTokenizer`（`weight`、`bias`、`cls` 参数；`forward` 返回 `(B, n_features+1, d_token)` 张量）。

> 能力标签：**GA1** · threshold ≥ 0.7